In [0]:
%sql
use catalog pl_workforce_analysis;
use schema silver;

In [0]:
accounts_df = spark.table("pl_workforce_analysis.bronze.accounts")
company_df = spark.table("pl_workforce_analysis.bronze.company")
department_df = spark.table("pl_workforce_analysis.bronze.departments")
employee_df = spark.table("pl_workforce_analysis.bronze.employees")
payroll_df = spark.table("pl_workforce_analysis.bronze.payroll")
gl_df = spark.table("pl_workforce_analysis.bronze.general_ledgers")

In [0]:
accounts_clean = accounts_df.dropDuplicates()
company_clean = company_df.dropDuplicates()
departments_clean = department_df.dropDuplicates()
employees_clean = employee_df.dropDuplicates()
payroll_clean = payroll_df.dropDuplicates()
gl_clean = gl_df.dropDuplicates()



In [0]:
from pyspark.sql.functions import upper, trim, when, col
from pyspark.sql.types import BooleanType

accounts_clean = accounts_clean.withColumn("account_code", trim("account_code")).withColumn("account_name", trim("account_name")) \
    .withColumn("account_type", upper(trim("account_type"))).withColumn("category", trim("category"))

accounts_clean = accounts_clean.withColumn(
    "pnl_sign",when(col("account_type") == "REVENUE", 1).when(col("account_type") == "CONTRA REVENUE", -1).when(col("account_type") == "COGS", -1)
    .when(col("account_type") == "CONTRA COGS", 1)  .when(col("account_type") == "EXPENSE", -1).otherwise(0))

accounts_clean = accounts_clean.withColumn("is_active",accounts_clean["is_active"].cast(BooleanType()))

accounts_clean = accounts_clean.withColumn(
    "pnl_section",when(col("account_type") == "REVENUE", "Revenue")
    .when(col("account_type") =="CONTRA REVENUE", "Revenue")
    .when(col("account_type") =="COGS","COGS")
    .when(col("account_type") == "CONTRA COGS","COGS")
    .when(col("account_type") == "EXPENSE","Operating Expense")
    .otherwise("Other")
)

display(accounts_clean)

In [0]:
accounts_clean.write.mode("overwrite").saveAsTable("pl_workforce_analysis.silver.accounts")


In [0]:
departments_clean = departments_clean.withColumn("department_code", trim(col("department_code"))) \
    .withColumn("department_name", trim(col("department_name"))) \
    .withColumn("cost_center", trim(col("cost_center")))

departments_clean = departments_clean.filter(col("department_id").isNotNull() & col("department_code").isNotNull())

display(departments_clean)


In [0]:
departments_clean.write.mode("overwrite").saveAsTable("pl_workforce_analysis.silver.departments")


In [0]:
from pyspark.sql.functions import col, trim, upper, col, to_date, concat_ws
from pyspark.sql.types import DoubleType, BooleanType

employees_clean = employees_clean\
    .withColumn("first_name", trim(col("first_name")))\
    .withColumn("last_name", trim(col("last_name")))\
    .withColumn("email", trim(col("email")))\
    .withColumn("position", trim(col("position")))

employees_clean = employees_clean.withColumn("full_name",concat_ws(" ", col("first_name"), col("last_name")))

employees_clean = employees_clean.withColumn("position",upper(col("position")))

employees_clean = employees_clean.withColumn("base_salary",col("base_salary").cast(DoubleType()))

employees_clean = employees_clean.withColumn("is_active",col("is_active").cast(BooleanType()))

employees_clean = employees_clean \
    .withColumn(
        "hire_date",
        to_date(employees_clean['hire_date'], "dd-MM-yyyy HH:mm")
    ) \
    .withColumn(
        "termination_date",
        to_date(employees_clean['termination_date'], "dd-MM-yyyy HH:mm")
    )

display(employees_clean)


In [0]:
employees_clean.write.mode("overwrite").saveAsTable("pl_workforce_analysis.silver.employees")

In [0]:
from pyspark.sql.functions import to_date
from pyspark.sql.functions import trim, col

gl_clean = gl_clean.withColumn("reference_number", trim(col("reference_number"))).withColumn("description", trim(col("description"))).withColumn("transaction_type", trim(col("transaction_type"))).withColumn("status", trim(col("status")))

gl_clean = gl_clean \
    .withColumn(
        "entry_date",
        to_date(gl_clean['entry_date'], "dd-MM-yyyy HH:mm")
    ) \
    .withColumn(
        "posting_date",
        to_date(gl_clean['posting_date'], "dd-MM-yyyy HH:mm")
    )

gl_clean = gl_clean.withColumn("debit_amount",gl_clean["debit_amount"].cast(DoubleType()))

gl_clean = gl_clean.withColumn("credit_amount",gl_clean["credit_amount"].cast(DoubleType()))

gl_clean = gl_clean.withColumn("amount",gl_clean["debit_amount"]-gl_clean["credit_amount"])

gl_clean = gl_clean.withColumn("amount",when(gl_clean["status"] == "Reversed", 0).otherwise(gl_clean["amount"])) 

display(gl_clean)


In [0]:
gl_clean.write.mode("overwrite").saveAsTable("pl_workforce_analysis.silver.general_ledgers")


In [0]:
from pyspark.sql.functions import col, split, to_date, when

numeric_cols = [
    "gross_salary", "bonus", "overtime_pay", "commission", "allowances",
    "tax_deduction", "social_security", "health_insurance",
    "retirement_contribution", "other_deductions", "net_salary"
]
for c in numeric_cols:
    payroll_clean = payroll_clean.withColumn(c, col(c).cast("double"))

payroll_clean = payroll_clean.withColumn("payment_method", trim(col("payment_method"))).withColumn("status", trim(col("status")))

payroll_clean = payroll_clean \
    .withColumn(
        "pay_date",
        to_date(payroll_clean['pay_date'], "dd-MM-yyyy HH:mm")
    ) \
    .withColumn(
        "pay_period_start",
        to_date(payroll_clean['pay_period_start'], "dd-MM-yyyy HH:mm")
    ) \
    .withColumn(
        "pay_period_end",
        to_date(payroll_clean['pay_period_end'], "dd-MM-yyyy HH:mm")
    )

payroll_clean = payroll_clean.filter(col("status") == "Paid")

payroll_df = payroll_df.drop("gross_salary").drop("net_salary")

payroll_df = payroll_df.withColumn(
    "net_salary",(col("base_salary") + col("bonus") + col("overtime_pay") + col("commission") + col("allowances")) - ( col("tax_deduction") + col("social_security") + col("health_insurance") + col("retirement_contribution") + col("other_deductions"))
)

payroll_df = payroll_df.withColumn(
    "gross_salary", col("base_salary") + col("bonus") + col("overtime_pay") + col("commission") + col("allowances")
)

payroll_clean = payroll_clean.withColumn(
    "total_payroll_cost",col("gross_salary") + col("bonus") + col("overtime_pay") + col("commission") + col("allowances")
)

payroll_clean = payroll_clean.withColumn(
    "total_deductions",col("tax_deduction") + col("social_security") +  col("health_insurance") + col("retirement_contribution") + col("other_deductions")
)

display(payroll_clean)




In [0]:
payroll_clean.write.mode("overwrite").saveAsTable("pl_workforce_analysis.silver.payroll")

In [0]:
from pyspark.sql.functions import col, trim, to_date
from pyspark.sql.types import BooleanType

company_clean = spark.table("pl_workforce_analysis.bronze.company")

company_clean = company_clean.withColumn("company_name", trim(col("company_name"))).withColumn("industry", trim(col("industry"))).withColumn("country", trim(col("country")))

company_clean = company_clean.withColumn("established_date",to_date(company_clean['established_date'], "dd-MM-yyyy"))

company_clean = company_clean.withColumn("is_active",col("is_active").cast(BooleanType()))

display(company_clean)


In [0]:
company_clean.write.mode("overwrite").saveAsTable("pl_workforce_analysis.silver.company")